<a href="https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: Content vs. performance alignment** — finding pages where how they *rank* and how they *perform* pull in different directions from what the query intent behind them would predict.

**Task type: scoring.** I want a continuous **alignment score** per page, not a yes/no label. "Aligned" and "misaligned" aren't two clean buckets — a page can be slightly off, badly off, or fine. A classifier forces a threshold I'd have to invent by hand (which is exactly the fixed-rule problem this lane is supposed to move past). A score lets me **rank** pages by how much their observed performance diverges from what their ranking position would lead you to expect, and hand the top of that ranked list to a human for review — which is the actual decision this needs to support (what to look at first).

Clustering doesn't fit: I'm not grouping similar pages, I'm measuring a gap for each one. Classification is possible as a *later* simplification (e.g. "top 10% divergence = flag"), but the underlying task is scoring.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CHECK: load the starter slice once so every later section can reuse it.
# Tries local files first, then downloads the CSV from the GitHub repo when needed.
import pandas as pd
from pathlib import Path

CSV_URL = "https://raw.githubusercontent.com/SubhadeepBhadra/subhflyrank-internship/main/data/raw/content_refresh_anonymized.csv"

CANDIDATE_PATHS = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/subhflyrank-internship/data/raw/content_refresh_anonymized.csv",
]

data_path = next((p for p in CANDIDATE_PATHS if Path(p).exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    print(f"Loaded from local path: {data_path}")
else:
    try:
        df = pd.read_csv(CSV_URL)
        print(f"Loaded from GitHub URL: {CSV_URL}")
    except Exception as e:
        raise FileNotFoundError(
            f"Couldn't find the CSV locally or download it from GitHub. Error: {e}"
        ) from e

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Columns:", list(df.columns))

Loaded from GitHub URL: https://raw.githubusercontent.com/SubhadeepBhadra/subhflyrank-internship/main/data/raw/content_refresh_anonymized.csv
Shape: 30,000 rows x 44 columns
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There's no column in this data that says "misaligned" — nobody has ever labeled that. So this is a **proxy**, built from a defined rule, not an observed outcome.

**Proxy: `alignment_gap`** = a page's actual click-through rate minus the *typical* CTR for pages sitting at that same search position. Position sets an expectation (position 3 usually gets far more clicks than position 13, regardless of content). A page whose real CTR falls well **below** what its position would predict is a candidate for "ranks fine, but the content/query match is off" — a divergence between where it ranks and how it performs. A page well **above** expectation is the mirror case (over-performing its position — good, but also a mismatch worth understanding).

The "typical CTR for a position" benchmark itself is *observed* (computed straight from the data, not asserted) — the rule is only in how I turn that benchmark into a gap score. That distinction — observed benchmark, defined-rule label — is what I'll flag in every write-up so nobody reads this as ground truth.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CHECK: find the columns we need by keyword, since exact names can vary by export,
# then build the position-expected-CTR benchmark and the alignment_gap proxy.
import numpy as np

def find_col(df, *keywords):
    for col in df.columns:
        lc = col.lower()
        if all(k in lc for k in keywords):
            return col
    return None

ctr_col = find_col(df, "ctr")
position_col = find_col(df, "position") or find_col(df, "rank")

print("CTR column guess:", ctr_col)
print("Position column guess:", position_col)

if ctr_col and position_col:
    work = df[[ctr_col, position_col]].dropna().copy()
    work["position_bucket"] = work[position_col].round().clip(1, 30)
    expected_ctr = work.groupby("position_bucket")[ctr_col].median()
    work["expected_ctr"] = work["position_bucket"].map(expected_ctr)
    work["alignment_gap"] = work[ctr_col] - work["expected_ctr"]
    print("\nExpected CTR by position (first few buckets):")
    print(expected_ctr.head())
    print(f"\nalignment_gap built for {len(work):,} rows. Example:")
    print(work.head())
else:
    print("\nCouldn't auto-detect both columns — check the data dictionary and set "
          "ctr_col / position_col manually before re-running.")

CTR column guess: ctr
Position column guess: avg_position

Expected CTR by position (first few buckets):
position_bucket
1.0    0.00
2.0    0.03
3.0    0.16
4.0    0.23
5.0    0.22
Name: ctr, dtype: float64

alignment_gap built for 30,000 rows. Example:
    ctr  avg_position  position_bucket  expected_ctr  alignment_gap
0  0.76          10.6             11.0          0.12           0.64
1  0.05          20.3             20.0          0.10          -0.05
2  0.09          36.5             30.0          0.00           0.09
3  0.49           6.2              6.0          0.17           0.32
4  0.13          44.0             30.0          0.00           0.13


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@25 on a manually-reviewed sample.** Take the 25 pages with the most negative `alignment_gap` (biggest underperformance-for-position), have a human actually look at them, and count how many turn out to be real content/intent mismatches versus noise (seasonal dip, tiny sample size, a one-off SERP feature eating clicks, etc.).

I'm defending this metric over something automatic like RMSE because there's no ground-truth label to regress against — the proxy *is* the model's opinion, not the truth. Precision@K against human judgment is the honest way to check whether the top of the ranked list is actually useful, and it directly matches the real decision this supports: which pages does a human look at first. "Good" = most of the top 25 are judged worth a second look, not wasted attention.

This is a **decision-support** metric, not a claim about causality — a high score means "worth reviewing," never "this page is definitely broken."

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CHECK: build the review queue and the precision function — leave the actual
# score blank until a human has reviewed the queue (no manufactured numbers).

def precision_at_k(reviewed_flags):
    """reviewed_flags: list of True/False, one per reviewed row,
    True = human confirmed a real content/performance mismatch."""
    if not reviewed_flags:
        return None
    return sum(reviewed_flags) / len(reviewed_flags)

if "alignment_gap" in work.columns:
    review_queue = work.sort_values("alignment_gap").head(25)
    print(f"Review queue built: {len(review_queue)} pages queued for manual review.")
    print(review_queue[[position_col, ctr_col, "expected_ctr", "alignment_gap"]].head())

    # TODO once reviewed: reviewed_flags = [True, False, True, ...]  (len == 25)
    # precision_at_k(reviewed_flags)
    print("\nPrecision@25: not yet computed — pending manual review of review_queue.")
else:
    print("alignment_gap not available — rerun Section 2's check first.")

Review queue built: 25 pages queued for manual review.
       avg_position  ctr  expected_ctr  alignment_gap
24074           4.0  0.0          0.23          -0.23
28725           4.0  0.0          0.23          -0.23
17766           3.7  0.0          0.23          -0.23
17767           4.1  0.0          0.23          -0.23
25889           4.2  0.0          0.23          -0.23

Precision@25: not yet computed — pending manual review of review_queue.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


**One row = one page** (this content-refresh slice is already page-level, not page-query-day level — confirmed by checking for duplicate rows below). Every alignment_gap score lives at the page level, which matches the decision it supports: "review this page," not "review this page for this one query on this one day."

In [4]:
# CHECK: confirm the row grain and show the unit of analysis as a real dataframe.
print(f"Total rows: {len(df):,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

id_col = find_col(df, "id") or find_col(df, "page") or find_col(df, "url")
if id_col:
    print(f"Unique values in likely id column '{id_col}': {df[id_col].nunique():,} "
          f"out of {len(df):,} rows")

df.head()

Total rows: 30,000
Duplicate rows: 0
Unique values in likely id column 'content_id': 30,000 out of 30,000 rows


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. "position 8–15 and CTR below 2%") only catches the exact combination someone already thought to write down. Two problems with that here:

1. **"Expected CTR" isn't one number — it's a whole distribution per position**, and that distribution itself has spread (shown below). A single CTR threshold either misses pages at positions where the whole neighborhood normally gets low clicks, or flags pages that are actually normal for their position. The expected-CTR benchmark has to be learned from the data's own shape, not asserted.
2. **Alignment likely depends on more than position and CTR** — content age, topic, query intent category, and engagement signals could all interact. A hand-written if-statement can encode maybe two or three conditions before it becomes unreadable; a learned scorer can weigh many signals at once and update as more data comes in, instead of needing manual re-tuning every time the pattern shifts.

The check below shows problem #1 directly: CTR varies a lot even *within* a single position bucket, which is exactly the messiness a single-threshold rule can't capture.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CHECK: how much does CTR vary within a single position bucket?
# High spread = a single CTR threshold per position can't cleanly separate
# "aligned" from "misaligned" pages.
if "alignment_gap" in work.columns:
    spread = work.groupby("position_bucket")[ctr_col].agg(["median", "std", "count"])
    spread = spread[spread["count"] >= 10]  # ignore buckets too small to be meaningful
    spread["coefficient_of_variation"] = spread["std"] / spread["median"]
    print("CTR spread within each position bucket (bigger std/median = messier):")
    print(spread.sort_values("coefficient_of_variation", ascending=False).head(10))
    print(f"\nMedian coefficient of variation across buckets: "
          f"{spread['coefficient_of_variation'].median():.2f}")
else:
    print("alignment_gap not available — rerun Section 2's check first.")

CTR spread within each position bucket (bigger std/median = messier):
                 median       std  count  coefficient_of_variation
position_bucket                                                   
1.0                0.00  8.142423   1414                       inf
30.0               0.00  1.773228   4772                       inf
2.0                0.03  9.849578    540                328.319254
26.0               0.06  4.328316    420                 72.138593
25.0               0.08  5.586015    359                 69.825185
3.0                0.16  4.816500    737                 30.103123
8.0                0.10  2.802422   2072                 28.024217
9.0                0.12  3.175370   1272                 26.461417
21.0               0.07  1.718489    429                 24.549845
4.0                0.23  5.102116   1460                 22.183115

Median coefficient of variation across buckets: 16.12


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.